# 04 — Sentinel Suite: Data Drift & Model Health

**P.U.L.S.E.**

Monitors the NHANES diabetes model using Evidently 0.7.

Compares:
- **Reference**: NHANES 2015-16  
- **Current**: NHANES 2021-23

Covers:
1. Distribution shift visualisation (5 clinical features)
2. Evidently drift report generation
3. Health alert tier system (CRITICAL / WARNING / INFO)
4. Model AUC stability across cohorts

In [ ]:
import os, sys
ROOT = os.path.dirname(os.path.abspath('.'))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

import warnings
warnings.filterwarnings('ignore')

import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_auc_score

sns.set_theme(style='whitegrid', palette='muted')
DATA      = os.path.join(ROOT, 'data', 'processed')
MODEL_DIR = os.path.join(ROOT, 'models')
print('Ready')

## 1  Load NHANES cohorts

In [ ]:
ref = pd.read_parquet(os.path.join(DATA, 'nhanes_2015.parquet'))
cur = pd.read_parquet(os.path.join(DATA, 'nhanes_2021.parquet'))

DRIFT_FEATS = ['age', 'gender', 'glucose', 'hba1c', 'serum_creatinine']

print(f'Reference: {ref.shape[0]:,} rows   Current: {cur.shape[0]:,} rows')

## 2  Distribution comparison — side-by-side histograms

In [ ]:
fig, axes = plt.subplots(1, len(DRIFT_FEATS), figsize=(18, 4))

for ax, feat in zip(axes, DRIFT_FEATS):
    r = ref[feat].dropna()
    c = cur[feat].dropna()
    bins = np.linspace(min(r.min(), c.min()), max(r.max(), c.max()), 30)
    ax.hist(r, bins=bins, alpha=0.6, label='2015-16', color='steelblue', density=True)
    ax.hist(c, bins=bins, alpha=0.6, label='2021-23', color='tomato',    density=True)
    ax.set_title(feat)
    ax.legend(fontsize=7)

fig.suptitle('NHANES Feature Distributions: 2015-16 vs 2021-23', fontsize=13)
plt.tight_layout()
plt.show()

## 3  Run Evidently drift report

In [ ]:
from src.monitoring.drift_detector import run_drift_report
from src.monitoring.health_report  import build_health_report, console_render

result  = run_drift_report()
report  = build_health_report(result)
console_render(report)

## 4  Drift distances — bar chart

In [ ]:
drift_df = pd.DataFrame(
    [(f, v['distance'], v['drifted'])
     for f, v in report['feature_drift'].items()],
    columns=['Feature', 'Wasserstein Distance', 'Drifted']
).sort_values('Wasserstein Distance', ascending=False)

colours = ['#D32F2F' if d else '#388E3C' for d in drift_df['Drifted']]

fig, ax = plt.subplots(figsize=(7, 4))
ax.barh(drift_df['Feature'], drift_df['Wasserstein Distance'], color=colours)
ax.axvline(0.10, color='orange', linestyle='--', lw=1.2, label='WARNING (0.10)')
ax.axvline(0.20, color='red',    linestyle='--', lw=1.2, label='CRITICAL (0.20)')
ax.set_xlabel('Wasserstein / Jensen-Shannon distance')
ax.set_title('Feature Drift — NHANES 2015-16 → 2021-23')
ax.legend()
plt.tight_layout()
plt.show()

## 5  Model AUC stability — 2015-16 vs 2021-23

In [ ]:
with open(os.path.join(MODEL_DIR, 'nhanes_diab_xgb.pkl'), 'rb') as f:
    model = pickle.load(f)
with open(os.path.join(MODEL_DIR, 'nhanes_diab_features.pkl'), 'rb') as f:
    features = pickle.load(f)

aucs = {}
for name, df in [('2015-16', ref), ('2021-23', cur)]:
    labelled = df.dropna(subset=['diabetes_label'])
    if len(labelled) < 10:
        print(f'{name}: insufficient labels')
        continue
    X = labelled[features]
    y = labelled['diabetes_label'].astype(int)
    proba = model.predict_proba(X)[:, 1]
    aucs[name] = roc_auc_score(y, proba)
    print(f'{name}: AUC = {aucs[name]:.4f}  (n={len(labelled):,})')

fig, ax = plt.subplots(figsize=(5, 3))
ax.bar(aucs.keys(), aucs.values(), color=['steelblue', 'tomato'])
ax.set_ylim(0.85, 1.0)
ax.set_ylabel('AUC')
ax.set_title('Model AUC Stability Across Cohorts')
for i, (k, v) in enumerate(aucs.items()):
    ax.text(i, v + 0.002, f'{v:.4f}', ha='center', fontweight='bold')
plt.tight_layout()
plt.show()